# Solar Filament Segmentation — Full Pipeline (Final Submission)

**Competition:** Solar Filament Segmentation Challenge 2026 (Kaggle, IEEE BigData 2026 / AURA-NSO)
**Team:** APS Team (Kaggle account `apsteam`)
**Selected final submission:** Kaggle ref **58252739** — public Panoptic Quality **0.38**

## Method (honest summary)
Single-stage instance segmentation at native resolution. A YOLO26x-seg model
(public checkpoint by **HDJoJo**, `yolo26x-seg-solar-filament`, trained on the 707
competition train images only) is run once over each 2048x2048 GONG H-alpha test
image at `imgsz=2048`, `conf=0.25`, `iou=0.0`, `max_det=100`. Predicted masks are
made mutually disjoint by score-ordered overlap removal (the evaluator rejects
instance sets with multi-covered pixels), masks smaller than 5 px are dropped,
and each surviving instance is written as a compressed COCO RLE row in
`submission.csv`. There is **no** detector->refiner cascade, **no** test-time
augmentation and **no** ensemble merge in the selected submission: every one of
those additions was built, measured on a held-out local validation split, and
rejected for failing to beat this configuration (see the technical report).

## Reproduce
1. `pip install -r requirements.txt`
2. Put the 180 competition test JPEGs in a folder (on Kaggle this is the
   competition mount; locally set the `FILAMENT_TEST_DIR` environment variable
   or edit `TEST_IMAGES_DIR` below).
3. Run this notebook top to bottom. The model weights are loaded from
   `weights/` in this repository — no retraining and no external download is
   required.
4. Output: `submission.csv` (1,435 rows for the selected run; 179/180 images
   covered — the model returns no instance above threshold for one image).

In [ ]:
import cv2
import csv
import os
import glob
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from pycocotools import mask as mask_utils
from ultralytics import YOLO

In [ ]:
# --- Configuration -----------------------------------------------------------
# Test images: on Kaggle the competition mount is used automatically; locally
# point FILAMENT_TEST_DIR at a folder holding the 180 test JPEGs.
TEST_IMAGES_DIR = os.environ.get(
    "FILAMENT_TEST_DIR",
    "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images",
)

# Model weights are stored in this repository (no re-training needed).
_REPO_WEIGHTS = os.path.join("weights", "yolo26x-seg-solar-filament-2048-1109.pt")
_KAGGLE_WEIGHTS = "/kaggle/input/models/hdjojo/yolo26x-seg-solar-filament/pytorch/default/1/yolo26x-seg-solar-filament-2048-1109.pt"
MODEL_PATH = _REPO_WEIGHTS if os.path.exists(_REPO_WEIGHTS) else _KAGGLE_WEIGHTS

class Config_YOLO26x():
    MODEL_PATH = MODEL_PATH
    IMG_SIZE = 2048
    CONF = 0.25
    IOU = 0.0
    MAX_DET = 100

CONFIG_YOLO26x = Config_YOLO26x()
model_yolo26x = YOLO(CONFIG_YOLO26x.MODEL_PATH)

In [ ]:
def mask_to_rle(binary_mask):
    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle["counts"]

def remove_instance_overlaps(masks, scores):

    sorted_indices = np.argsort(scores)[::-1]
    sorted_masks = masks[sorted_indices]

    accumulated_mask = np.zeros(masks.shape[1:], dtype=np.uint8)
    non_overlapping_masks = []

    for mask in sorted_masks:
        clean_mask = mask & ~accumulated_mask
        non_overlapping_masks.append(clean_mask)
        accumulated_mask |= mask

    return np.array(non_overlapping_masks)

In [ ]:
submission_rows = []
for img_path in tqdm(sorted(glob.glob(f"{TEST_IMAGES_DIR}/*.jpeg")), desc="Inference"):
    file_stem = os.path.splitext(os.path.basename(img_path))[0]
    result = model_yolo26x.predict(
        img_path, imgsz=CONFIG_YOLO26x.IMG_SIZE, conf=CONFIG_YOLO26x.CONF, iou=CONFIG_YOLO26x.IOU, max_det=CONFIG_YOLO26x.MAX_DET, verbose=False)[0]

    if result.masks is None: continue
    result_masks_data = result.masks.data.cpu().numpy()
    scores = result.boxes.cls.cpu().numpy()
    final_masks = remove_instance_overlaps(result_masks_data, scores)

    for i, m in enumerate(final_masks):
        m_res = cv2.resize(m.astype(np.uint8), (2048, 2048), interpolation=cv2.INTER_NEAREST)
        if m_res.sum() < 5: continue
        submission_rows.append({
            "filament_id": f"{file_stem}_{i+1}",
            "segmentation_rle": mask_to_rle(m_res)
        })

In [ ]:
sub_path = "submission.csv"
with open(sub_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["filament_id", "segmentation_rle"])
    writer.writeheader()
    writer.writerows(submission_rows)
print(f"Submission saved to {sub_path} ({len(submission_rows)} instances)")

In [ ]:
# Fail-closed validator (same checks as the filed run): the output must cover
# the test set and every sampled RLE must decode to a positive-area 2048x2048 mask.
import json as _json
from pycocotools import mask as _mu
_test_files = sorted(glob.glob(f'{TEST_IMAGES_DIR}/*.jpeg'))
assert len(_test_files) == 180, f'expected 180 test images, found {len(_test_files)}'
_df = pd.read_csv(sub_path)
assert list(_df.columns) == ['filament_id','segmentation_rle'], _df.columns.tolist()
_covered = set(_df['filament_id'].str.rsplit('_', n=1).str[0])
_stems = set(os.path.splitext(os.path.basename(p))[0] for p in _test_files)
_dec = 0
for _rle in _df['segmentation_rle'].iloc[::max(1, len(_df)//50)]:
    _m = _mu.decode({'size':[2048,2048],'counts':_rle.encode('utf-8')})
    assert _m.shape == (2048,2048) and _m.sum() > 0
    _dec += 1
_report = {'rows': int(len(_df)), 'test_images': 180, 'images_covered': len(_covered & _stems), 'images_missing': sorted(_stems - _covered), 'sample_rles_decoded_ok': _dec, 'model': 'hdjojo/yolo26x-seg-solar-filament pytorch default/1 (2048-1109)'}
with open('run_report.json','w') as _f: _json.dump(_report, _f, indent=1)
print('VALIDATOR PASS', _report)